# Población de Vilalba, Ordes y Salvaterra de Miño

**Objetivo:** comparar la evolución de la población, la proporción de hombres y mujeres, la población extranjera y la edad media de tres concellos gallegos con situaciones distintas:

| Concello | Provincia | Código INE |
|---|---|---|
| Vilalba | Lugo | 27065 |
| Ordes | A Coruña | 15059 |
| Salvaterra de Miño | Pontevedra | 36050 |

**Fuentes**
- **INE** – *Cifras oficiales de población de los municipios españoles: Revisión del Padrón Municipal* (API JSON, tablas por provincia).
- **IGE** – fichas municipales y banco de datos (edad media y población extranjera).

**Qué hace este notebook**
1. Descarga del INE la población total, de hombres y de mujeres (toda la serie disponible).
2. Limpia los datos y los deja en formato *tidy*.
3. Comprueba la calidad (hombres + mujeres = total, años sin huecos…).
4. Calcula los indicadores: índice base 100, % de mujeres y ratio de masculinidad.
5. Añade la edad media y la población extranjera (archivo `data/manual/`).
6. Guarda los datos limpios en `data/processed/` para usarlos luego en la app de Streamlit.

## 1. Configuración

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import requests
import plotly.express as px

# Carpeta raíz del proyecto 
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"
MANUAL = RAIZ / "data" / "manual"
PROCESSED = RAIZ / "data" / "processed"
for carpeta in (RAW, PROCESSED):
    carpeta.mkdir(parents=True, exist_ok=True)

# Concellos a comparar: aparece en el INE
CONCELLOS = {
    "Vilalba":            {"codigo_ine": "27065", "provincia": "Lugo",       "tabla_ine": 2880},
    "Ordes":              {"codigo_ine": "15059", "provincia": "A Coruña",   "tabla_ine": 2868},
    "Salvaterra de Miño": {"codigo_ine": "36050", "provincia": "Pontevedra", "tabla_ine": 2890},
}

ANYO_BASE = 2010     # año que vale 100 en el índice base 100

pd.set_option("display.max_columns", 20)
print("Carpeta del proyecto:", RAIZ)

Carpeta del proyecto: c:\Users\Hugo\Downloads\concellos-galicia\concellos-galicia


## 2. Descarga desde la API del INE

Cada provincia tiene su tabla de *Población por municipios y sexo*: A Coruña = 2868, Lugo = 2880, Pontevedra = 2890.

La URL sigue el patrón `https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{id}?nult=40` (los últimos 40 periodos, es decir, toda la serie) y devuelve una lista de **series**. Cada serie tiene un `Nombre` (p. ej. `"Ordes. Hombres. Total habitantes. Personas."`) y una lista `Data` con un valor por año.

La respuesta se guarda en `data/raw/` para no repetir la descarga cada vez que ejecutes el notebook. Si quieres forzar una descarga nueva, borra el `.json` o usa `forzar=True`.

In [ ]:
URL_INE = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{tabla}"

def descargar_tabla_ine(tabla: int, n_periodos: int = 40, forzar: bool = False) -> list:
    """Descarga una tabla del INE (JSON) y la guarda en data/raw/. Si ya existe, la lee del disco."""
    ruta = RAW / f"ine_tabla_{tabla}_serie_completa.json"
    if ruta.exists() and not forzar:
        return json.loads(ruta.read_text(encoding="utf-8"))

    url = URL_INE.format(tabla=tabla)
    params = {"nult": n_periodos}  # últimos 40 años disponibles = serie completa
    respuesta = requests.get(url, params=params, timeout=60)
    respuesta.raise_for_status()
    datos = respuesta.json()
    ruta.write_text(json.dumps(datos, ensure_ascii=False), encoding="utf-8")
    print(f"Tabla {tabla} descargada: {len(datos)} series -> {ruta.name}")
    return datos


tablas = sorted({c["tabla_ine"] for c in CONCELLOS.values()})
raw_ine = {t: descargar_tabla_ine(t) for t in tablas}
{t: len(series) for t, series in raw_ine.items()}

Tabla 2868 descargada: 288 series -> ine_tabla_2868_serie_completa.json
Tabla 2880 descargada: 204 series -> ine_tabla_2880_serie_completa.json
Tabla 2890 descargada: 192 series -> ine_tabla_2890_serie_completa.json


{2868: 288, 2880: 204, 2890: 192}

## 3. Limpieza: de JSON a tabla *tidy*

Del `Nombre` de cada serie sacamos el **municipio** y el **sexo** con una expresión regular. Nos quedamos solo con los tres concellos y lo pasamos a formato largo:

`codigo_ine | concello | anyo | indicador | sexo | valor`

In [ ]:
PATRON_NOMBRE = re.compile(r"^(?P<municipio>.+?)\.\s(?P<sexo>Total|Hombres|Mujeres)\.\s")

def series_a_dataframe(series: list) -> pd.DataFrame:
    filas = []
    for s in series:
        m = PATRON_NOMBRE.match(s["Nombre"])
        if not m or m["municipio"] not in CONCELLOS:
            continue
        for d in s["Data"]:
            if d.get("Valor") is None:
                continue
            filas.append({
                "concello": m["municipio"],
                "sexo": m["sexo"].lower(),
                "anyo": int(d["Anyo"]),
                "valor": float(d["Valor"]),
            })
    return pd.DataFrame(filas)


poblacion = pd.concat([series_a_dataframe(s) for s in raw_ine.values()], ignore_index=True)
poblacion["codigo_ine"] = poblacion["concello"].map(lambda c: CONCELLOS[c]["codigo_ine"])
poblacion["indicador"] = "poblacion"
poblacion["valor"] = poblacion["valor"].astype(int)
poblacion = (poblacion[["codigo_ine", "concello", "anyo", "indicador", "sexo", "valor"]]
             .sort_values(["concello", "anyo", "sexo"])
             .reset_index(drop=True))

faltan = set(CONCELLOS) - set(poblacion["concello"])
assert not faltan, f"No se encontraron en el INE: {faltan}. Revisa que el nombre coincide exactamente."

poblacion.head(9)

,codigo_ine,concello,anyo,indicador,sexo,valor
0,15059,Ordes,1996,poblacion,hombres,5802
1,15059,Ordes,1996,poblacion,mujeres,6091
2,15059,Ordes,1996,poblacion,total,11893
3,15059,Ordes,1998,poblacion,hombres,5782
4,15059,Ordes,1998,poblacion,mujeres,6079
5,15059,Ordes,1998,poblacion,total,11861
6,15059,Ordes,1999,poblacion,hombres,5820
7,15059,Ordes,1999,poblacion,mujeres,6126
8,15059,Ordes,1999,poblacion,total,11946


## 4. Control de calidad

Antes de analizar, comprobamos que los datos tienen sentido. Es justo lo que se valora en un proyecto de datos:
- ¿Hombres + mujeres = total en todos los años?
- ¿Hay años que faltan?
- ¿Hay saltos raros (variaciones de más del 5 % en un año)?

In [ ]:
ancho = poblacion.pivot_table(index=["concello", "anyo"], columns="sexo", values="valor").reset_index()
ancho[["hombres", "mujeres", "total"]] = ancho[["hombres", "mujeres", "total"]].astype(int)

# 1) Coherencia hombres + mujeres = total
ancho["diferencia"] = ancho["hombres"] + ancho["mujeres"] - ancho["total"]
incoherentes = ancho[ancho["diferencia"] != 0]
print(f"Filas donde hombres + mujeres ≠ total: {len(incoherentes)}")

# 2) Años que faltan en cada concello
for concello, grupo in ancho.groupby("concello"):
    anyos = set(grupo["anyo"])
    huecos = sorted(set(range(min(anyos), max(anyos) + 1)) - anyos)
    print(f"{concello}: {min(anyos)}–{max(anyos)}, años que faltan: {huecos or 'ninguno'}")

# 3) Variaciones anuales grandes (posibles cambios de método o errores)
ancho = ancho.sort_values(["concello", "anyo"])
ancho["var_anual_%"] = ancho.groupby("concello")["total"].pct_change() * 100
saltos = ancho[ancho["var_anual_%"].abs() > 5]
print(f"\nSaltos de más del 5 % en un año: {len(saltos)}")
saltos[["concello", "anyo", "total", "var_anual_%"]]

Filas donde hombres + mujeres ≠ total: 0
Ordes: 1996–2025, años que faltan: [1997]
Salvaterra de Miño: 1996–2025, años que faltan: [1997]
Vilalba: 1996–2025, años que faltan: [1997]

Saltos de más del 5 % en un año: 0


sexo,concello,anyo,total,var_anual_%


> **Nota sobre las fuentes:** el INE dejó de publicar la *Estadística del Padrón continuo* (con detalle por edad y nacionalidad) después del dato de 1 de enero de 2022. Desde entonces el detalle viene del *Censo anual de población*. Las **cifras oficiales** de población total por sexo que usamos aquí se siguen publicando cada año, pero si combinas tablas detalladas de antes y después de 2022 vigila si aparecen saltos.

## 5. Indicadores

In [ ]:
ind = ancho[["concello", "anyo", "total", "hombres", "mujeres"]].copy()
ind["codigo_ine"] = ind["concello"].map(lambda c: CONCELLOS[c]["codigo_ine"])

# % de mujeres y ratio de masculinidad (hombres por cada 100 mujeres)
ind["pct_mujeres"] = (ind["mujeres"] / ind["total"] * 100).round(2)
ind["ratio_masculinidad"] = (ind["hombres"] / ind["mujeres"] * 100).round(1)

# Índice base 100: la población del año base vale 100
base = ind[ind["anyo"] == ANYO_BASE].set_index("concello")["total"]
ind["indice_base100"] = (ind["total"] / ind["concello"].map(base) * 100).round(1)

# Variación respecto al año base
ind["var_vs_base_%"] = (ind["indice_base100"] - 100).round(1)

ultimo = ind["anyo"].max()
resumen = ind[ind["anyo"] == ultimo].set_index("concello")[
    ["total", "hombres", "mujeres", "pct_mujeres", "ratio_masculinidad", "var_vs_base_%"]
]
print(f"Resumen a 1 de enero de {ultimo} (variación respecto a {ANYO_BASE}):")
resumen

Resumen a 1 de enero de 2025 (variación respecto a 2010):


sexo,total,hombres,mujeres,pct_mujeres,ratio_masculinidad,var_vs_base_%
concello,,,,,,
Ordes,12990,6353,6637,51.09,95.7,0.9
Salvaterra de Miño,10617,5303,5314,50.05,99.8,12.3
Vilalba,13852,6709,7143,51.57,93.9,-9.6


## 6. Población extranjera (INE, Censo anual de población)

Las fichas municipales del IGE dan cifras de extranjeros que no cuadran (hombres + mujeres ≠ total), así que usamos directamente el **Censo anual de población del INE**: tabla 68537, *Población por sexo y nacionalidad (grandes grupos)*, para todos los municipios de España y los años 2021 a 2025.

**Extranjeros = población total − población de nacionalidad española.**

In [ ]:
URL_CENSO_NAC = "https://www.ine.es/jaxiT3/files/t/csv_bdsc/68537.csv"

def descargar_archivo(url: str, nombre: str, forzar: bool = False) -> Path:
    ruta = RAW / nombre
    if ruta.exists() and not forzar:
        return ruta
    print("Descargando", url, "(puede tardar un poco)...")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(ruta, "wb") as f:
            for trozo in r.iter_content(chunk_size=1 << 20):
                f.write(trozo)
    print(f"Guardado {ruta.name}: {ruta.stat().st_size / 1e6:.1f} MB")
    return ruta


def leer_csv_ine(ruta: Path) -> pd.DataFrame:
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=";", dtype=str, encoding=codificacion)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer {ruta}")


def buscar_columna(df: pd.DataFrame, texto: str) -> str:
    for c in df.columns:
        if texto.lower() in c.lower():
            return c
    raise KeyError(f"No hay ninguna columna con '{texto}'. Columnas: {list(df.columns)}")


censo = leer_csv_ine(descargar_archivo(URL_CENSO_NAC, "ine_censo_nacionalidad_68537.csv"))
print("Columnas del CSV:", list(censo.columns))

Descargando https://www.ine.es/jaxiT3/files/t/csv_bdsc/68537.csv (puede tardar un poco)...
Guardado ine_censo_nacionalidad_68537.csv: 102.9 MB
Columnas del CSV: ['Total Nacional', 'Provincias', 'Municipios', 'Sexo', 'País de nacionalidad', 'Periodo', 'Total']


In [ ]:
col_mun = buscar_columna(censo, "municip")
col_sexo = buscar_columna(censo, "sexo")
col_nac = buscar_columna(censo, "nacionalidad")
col_per = buscar_columna(censo, "periodo")
col_val = censo.columns[-1]  # la última columna es el valor

codigos = {c["codigo_ine"]: nombre for nombre, c in CONCELLOS.items()}
censo["codigo_ine"] = censo[col_mun].str.extract(r"^(\d{5})", expand=False)
nac = censo[censo["codigo_ine"].isin(codigos)].copy()

nac["concello"] = nac["codigo_ine"].map(codigos)
nac["anyo"] = nac[col_per].str.extract(r"(\d{4})", expand=False).astype(int)
nac["sexo"] = nac[col_sexo].str.strip().str.lower()
nac["nacionalidad"] = nac[col_nac].str.strip()
nac["valor"] = pd.to_numeric(
    nac[col_val].str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")

print("Categorías de nacionalidad:", sorted(nac["nacionalidad"].unique()))
print("Categorías de sexo:", sorted(nac["sexo"].unique()))

tabla = nac.pivot_table(index=["codigo_ine", "concello", "anyo", "sexo"],
                        columns="nacionalidad", values="valor", aggfunc="sum")
col_esp = next(c for c in tabla.columns if c.lower().startswith("españ"))
if "Total" in tabla.columns:
    total_censo = tabla["Total"]
else:  # si no hay columna Total, se suman todos los grupos de nacionalidad
    total_censo = tabla.sum(axis=1)

ext = pd.DataFrame({"total_censo": total_censo, "espanoles": tabla[col_esp]}).reset_index()
ext["extranjeros"] = ext["total_censo"] - ext["espanoles"]
ext["pct_extranjeros"] = (ext["extranjeros"] / ext["total_censo"] * 100).round(2)

# Control de calidad: hombres + mujeres = total
comp = ext.pivot_table(index=["concello", "anyo"], columns="sexo", values="extranjeros")
print("\nExtranjeros: filas donde hombres + mujeres ≠ total:",
      int(((comp["hombres"] + comp["mujeres"]) != comp["total"]).sum()))

ext[ext["sexo"] == "total"][["concello", "anyo", "total_censo", "extranjeros", "pct_extranjeros"]]

Categorías de nacionalidad: ['América del Norte', 'Apátridas', 'Asia', 'Centro América y Caribe', 'España', 'Oceanía', 'Resto de Europa', 'Sudamérica', 'Total', 'Unión Europea (sin España)', 'África']
Categorías de sexo: ['hombres', 'mujeres', 'total']

Extranjeros: filas donde hombres + mujeres ≠ total: 0


,concello,anyo,total_censo,extranjeros,pct_extranjeros
2,Ordes,2021,12593,618,4.91
5,Ordes,2022,12589,634,5.04
8,Ordes,2023,12663,797,6.29
11,Ordes,2024,12762,937,7.34
14,Ordes,2025,12970,1122,8.65
17,Vilalba,2021,14014,554,3.95
20,Vilalba,2022,13893,577,4.15
23,Vilalba,2023,13900,681,4.90
26,Vilalba,2024,13822,749,5.42
29,Vilalba,2025,13946,919,6.59


## 7. Edad media (IGE)

La edad media por concello la publica el IGE. El archivo `data/manual/extranjeros_edad_media.csv` trae los valores de las fichas municipales del IGE (aquí solo usamos las filas de **edad media**). Para completar la serie, descarga *Indicadores demográficos* del banco de datos del IGE y añade las filas con el mismo formato.

In [ ]:
manual = pd.read_csv(MANUAL / "extranjeros_edad_media.csv", dtype={"codigo_ine": str})
manual = manual[manual["indicador"] == "edad_media"]
edad = manual.pivot_table(index=["concello", "anyo"], columns="sexo", values="valor").reset_index()
edad

sexo,concello,anyo,hombres,mujeres,total
0,Ordes,2025,45.75,48.70,47.26
1,Salvaterra de Miño,2024,44.81,47.48,46.14
2,Salvaterra de Miño,2025,45.09,47.59,46.34
3,Vilalba,2023,49.13,51.89,50.55
4,Vilalba,2025,48.88,52.01,50.49


## 8. Guardar los datos limpios

In [ ]:
ext_tidy = pd.concat([
    ext.assign(indicador="extranjeros").rename(columns={"extranjeros": "valor"}),
    ext[ext["sexo"] == "total"].assign(indicador="pct_extranjeros").rename(columns={"pct_extranjeros": "valor"}),
], ignore_index=True)[["codigo_ine", "concello", "anyo", "indicador", "sexo", "valor"]]

tidy = pd.concat([poblacion, ext_tidy, manual.drop(columns="fuente")], ignore_index=True)
tidy.to_csv(PROCESSED / "concellos_tidy.csv", index=False)
ind.to_csv(PROCESSED / "indicadores_poblacion.csv", index=False)

print("Guardado:")
for f in sorted(PROCESSED.glob("*.csv")):
    print(" -", f.relative_to(RAIZ), f"({len(pd.read_csv(f))} filas)")

Guardado:
 - data\processed\concellos_tidy.csv (336 filas)
 - data\processed\indicadores_poblacion.csv (87 filas)


## 9. Primeros gráficos

In [ ]:
COLORES = {"Vilalba": "#2a78d6", "Ordes": "#eb6834", "Salvaterra de Miño": "#1baf7a"}

fig = px.line(ind[ind["anyo"] >= ANYO_BASE], x="anyo", y="indice_base100", color="concello",
              color_discrete_map=COLORES, markers=True,
              labels={"anyo": "Año", "indice_base100": f"Población ({ANYO_BASE} = 100)", "concello": ""},
              title=f"Evolución de la población (índice base 100, {ANYO_BASE} = 100)")
fig.add_hline(y=100, line_dash="dot", line_color="gray")
fig.show()

In [ ]:
fig = px.line(ind, x="anyo", y="total", color="concello", color_discrete_map=COLORES,
              labels={"anyo": "Año", "total": "Habitantes", "concello": ""},
              title="Población total (habitantes)")
fig.show()

In [ ]:
fig = px.line(ind, x="anyo", y="pct_mujeres", color="concello", color_discrete_map=COLORES,
              labels={"anyo": "Año", "pct_mujeres": "% de mujeres", "concello": ""},
              title="Porcentaje de mujeres sobre la población total")
fig.add_hline(y=50, line_dash="dot", line_color="gray")
fig.show()

In [ ]:
edad_ultima = edad.sort_values("anyo").groupby("concello").tail(1)
edad_largo = edad_ultima.melt(id_vars=["concello", "anyo"], value_vars=["hombres", "mujeres", "total"],
                              var_name="sexo", value_name="edad_media")
fig = px.bar(edad_largo, x="concello", y="edad_media", color="sexo", barmode="group",
             text="edad_media", labels={"concello": "", "edad_media": "Edad media (años)", "sexo": ""},
             title="Edad media por sexo (último año disponible)")
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_yaxes(range=[40, 55])
fig.show()

In [ ]:
pct = ext[ext["sexo"] == "total"]
fig = px.line(pct, x="anyo", y="pct_extranjeros", color="concello", color_discrete_map=COLORES, markers=True,
              labels={"anyo": "Año", "pct_extranjeros": "% de población extranjera", "concello": ""},
              title="Porcentaje de población extranjera (INE, Censo anual)")
fig.update_xaxes(dtick=1)
fig.show()